# Module 8 Wednesday Demo: Data Cleaning & Wrangling
## CS 5045: Computation for the Data Sciences  |  Virginia Tech

We walk through the full cleaning pipeline on `data.csv`, the same dataset used in the lecture and the assignment.

**Primary operations:** Filter (drop_duplicates) and Transform (everything else). Merge is introduced as a new technique at the end.

**How to follow along:** Run each cell in order. Read the markdown header before running the code cell below it.

**Data file:** `../Datasets/data.csv` relative to this notebook. If you copy `data.csv` into this folder, change the path to `data.csv`.

---
## Step 1: Inspect Before You Touch Anything

**Rule:** Before cleaning a single value, look at the raw data. You need to know what problems exist before you can fix them.

**Attempt question (answer in chat before running the cell below):**
What do you expect `isnull().sum()` to tell you? What would it mean if a column shows 0?

In [1]:
import pandas as pd

df = pd.read_csv('../Datasets/data.csv')

# Before you touch a single value, just look. Half of data cleaning is noticing the problem.
print('Shape:', df.shape)
print()
print('Column types:')
print(df.dtypes)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Unique values in key text columns:')
for col in ['gender', 'employed', 'smoker', 'health_rating']:
    print(f'  {col}: {sorted(df[col].dropna().unique())}')

Shape: (18, 12)

Column types:
response_id                 int64
age                       float64
gender                        str
income_usd                float64
education                     str
employed                      str
health_rating                 str
exercise_days_per_week      int64
smoker                        str
last_checkup_year           int64
zip_code                    int64
satisfaction_score          int64
dtype: object

Missing values per column:
response_id               0
age                       1
gender                    0
income_usd                1
education                 0
employed                  0
health_rating             0
exercise_days_per_week    0
smoker                    0
last_checkup_year         0
zip_code                  0
satisfaction_score        0
dtype: int64

Unique values in key text columns:
  gender: ['FEMALE', 'Female', 'MALE', 'Male', 'female', 'male']
  employed: ['No', 'YES', 'Yes', 'yes']
  smoker: ['N', 'NO', 'No', '

---
## Step 2: Standardize Text Categories (Transform)

**Problem:** The `gender`, `employed`, and `health_rating` columns have inconsistent capitalization: `'female'`, `'FEMALE'`, `'Female'` all mean the same thing but will be counted as three separate values.

**Fix:** Chain `.str.strip()` (removes invisible whitespace) and `.str.title()` (standardizes to title case).

**Discussion question:** Will `.str.title()` fix the `smoker` column? Why or why not?

In [2]:
# str.strip() removes invisible whitespace; str.title() standardizes capitalization
df['gender']        = df['gender'].str.strip().str.title()
df['employed']      = df['employed'].str.strip().str.title()
df['health_rating'] = df['health_rating'].str.strip().str.title()

# Three columns, one line of logic each -- that's the leverage string methods give you.
print('gender unique:       ', df['gender'].unique())
print('employed unique:     ', df['employed'].unique())
print('health_rating unique:', df['health_rating'].unique())
print()
# Why str.title() is NOT enough for smoker:
print('smoker still has:', df['smoker'].unique())
print('  --> "Y" and "N" survive str.title(); they need a lookup dictionary')

gender unique:        <StringArray>
['Female', 'Male']
Length: 2, dtype: str
employed unique:      <StringArray>
['Yes', 'No']
Length: 2, dtype: str
health_rating unique: <StringArray>
['Good', 'Fair', 'Excellent', 'Poor']
Length: 4, dtype: str

smoker still has: <StringArray>
['No', 'no', 'Yes', 'N', 'YES', 'Y', 'NO', 'yes']
Length: 8, dtype: str
  --> "Y" and "N" survive str.title(); they need a lookup dictionary


---
## Step 2b: Remap Abbreviations with map() (Transform)

A lookup dictionary maps every known variant to the canonical form.
Any value **not in the dictionary** becomes `NaN`, which surfaces hidden problems rather than silently leaving them.

---
## Step 3: Remove Duplicate Rows (Filter)

After remapping, remove exact duplicate rows.

**Discussion question:** Which rows are duplicates? How would their presence bias a mean satisfaction score?

In [3]:
# Remap smoker
# This dictionary is doing all the thinking here; map() just applies it, row by row.
smoker_map = {
    'Yes': 'Yes', 'yes': 'Yes', 'YES': 'Yes', 'Y': 'Yes',
    'No':  'No',  'no':  'No',  'NO':  'No',  'N': 'No',
}
df['smoker'] = df['smoker'].map(smoker_map)
print('smoker unique after map():', df['smoker'].unique())
print('smoker value counts:')
print(df['smoker'].value_counts())
print()

# Show duplicated rows before removing them
dupes = df[df.duplicated(keep=False)]
if len(dupes) > 0:
    print('Duplicated rows (shown once each side):')
    print(dupes[['response_id', 'age', 'gender', 'smoker']].to_string(index=False))
    print()

# Remove duplicates -- they never announce themselves, you have to go looking for them
print('Rows before dedup:', len(df))
df = df.drop_duplicates()
print('Rows after dedup: ', len(df))

smoker unique after map(): <StringArray>
['No', 'Yes']
Length: 2, dtype: str
smoker value counts:
smoker
No     12
Yes     6
Name: count, dtype: int64

Duplicated rows (shown once each side):
 response_id  age gender smoker
         101 34.0 Female     No
         107 29.0 Female    Yes
         101 34.0 Female     No
         107 29.0 Female    Yes

Rows before dedup: 18
Rows after dedup:  16


---
## Step 4: Handle Missing Values (Transform)

Two strategies:
- **Deletion:** drop rows with NaN. Safe when the dataset is large and missing values are random.
- **Imputation:** fill with an estimate. Median is more robust than mean when outliers are present.

---
## Step 5: Fix Data Types (Transform)

After filling NAs, convert `age` back to `int`. It was stored as `float64` because pandas cannot hold NaN in an integer column; once NAs are gone, the conversion is safe.

In [4]:
# Fill missing values with column medians
# Median, not mean, on purpose: one unusually high or low value should not drag this number around
med_age    = df['age'].median()
med_income = df['income_usd'].median()
print(f'Median age used to fill:    {med_age}')
print(f'Median income used to fill: {med_income}')

df['age']        = df['age'].fillna(med_age)
df['income_usd'] = df['income_usd'].fillna(med_income)

# Convert age back to int
df['age'] = df['age'].astype(int)

print()
print('Missing values after filling:', df.isnull().sum().sum())
print('age dtype:', df['age'].dtype)
print('Sample ages:', df['age'].values[:6])

Median age used to fill:    39.0
Median income used to fill: 53000.0

Missing values after filling: 0
age dtype: int64
Sample ages: [34 28 45 52 39 39]


---
## Step 6: Derive a New Column with apply() (Transform)

`apply()` runs a function on every row value. You define the logic; pandas handles the loop.
This is the most flexible Transform in pandas because the function can contain any Python logic.

In [5]:
def classify_age(age):
    # Plain Python -- nothing pandas-specific happens until we call apply() below
    if age < 35:
        return 'Young Adult'
    elif age < 50:
        return 'Middle Aged'
    else:
        return 'Older Adult'

df['age_group'] = df['age'].apply(classify_age)

print(df[['response_id', 'age', 'age_group']].to_string(index=False))

 response_id  age   age_group
         101   34 Young Adult
         102   28 Young Adult
         103   45 Middle Aged
         104   52 Older Adult
         105   39 Middle Aged
         106   39 Middle Aged
         107   29 Young Adult
         109   61 Older Adult
         110   33 Young Adult
         111   47 Middle Aged
         112   55 Older Adult
         113   22 Young Adult
         114   38 Middle Aged
         115   44 Middle Aged
         117   50 Older Adult
         118   36 Middle Aged


---
## Aggregate on Clean Data (Aggregate)

With a clean dataset, grouped analysis is trustworthy.
Each respondent appears exactly once; every category value is canonical.
The result below would have been wrong before we cleaned the data.

In [6]:
# This number only means something because of every step we took above it
summary = (
    df.groupby('age_group')['satisfaction_score']
      .mean()
      .round(2)
)
print('Mean satisfaction score by age group:')
print(summary)

Mean satisfaction score by age group:
age_group
Middle Aged    7.0
Older Adult    7.5
Young Adult    6.4
Name: satisfaction_score, dtype: float64


---
## Step 7: Merge Two Tables with pd.merge() (new technique)

Real datasets rarely live in a single table. A **merge** (also called a join) combines two tables on a shared key column.

Here, `df` has a `zip_code` column. A small reference table maps each ZIP code to a neighborhood name.

```
pd.merge(left, right, on='key_column', how='inner')
```

- **left:** the main table (`df`)
- **right:** the reference table (`zip_ref`)
- **on:** the column both tables share (`zip_code`)
- **how='inner':** keep only rows where the key appears in **both** tables

This is the same idea as a SQL INNER JOIN. Module 9 will use it more. Your project pipeline will almost certainly need it.

In [7]:
# Build the reference table inline -- only 3 of the 4 ZIP codes intentionally
# so the inner join drops rows 109, 113, 118 (zip 24063 not covered).
# This demos inner join row-dropping. Note: hw8.ipynb bonus loads data2.csv
# which covers all 4 ZIP codes, so no rows drop in the assignment.
zip_ref = pd.DataFrame({
    'zip_code':     [24060, 24061, 24062],
    'neighborhood': ['Blacksburg', 'Blacksburg (VT Campus)', 'Prices Fork']
})

print('Reference table (3 of 4 ZIP codes):')
print(zip_ref)
print()

# Same shared-key idea you'll see formalized when we get to SQL in Module 11
# Inner join: keep rows where zip_code appears in both tables
df_merged = pd.merge(df, zip_ref, on='zip_code', how='inner')

print(f'Rows before merge: {len(df)}')
print(f'Rows after merge:  {len(df_merged)}  (inner join drops any zip not in zip_ref)')
print()
print('Sample rows with new neighborhood column:')
print(df_merged[['response_id', 'zip_code', 'neighborhood', 'age_group', 'satisfaction_score']]
      .head(5).to_string(index=False))

Reference table (3 of 4 ZIP codes):
   zip_code            neighborhood
0     24060              Blacksburg
1     24061  Blacksburg (VT Campus)
2     24062             Prices Fork

Rows before merge: 16
Rows after merge:  13  (inner join drops any zip not in zip_ref)

Sample rows with new neighborhood column:
 response_id  zip_code           neighborhood   age_group  satisfaction_score
         101     24060             Blacksburg Young Adult                   8
         102     24061 Blacksburg (VT Campus) Young Adult                   7
         103     24060             Blacksburg Middle Aged                   5
         104     24062            Prices Fork Older Adult                   9
         105     24061 Blacksburg (VT Campus) Middle Aged                   8


---
## HW Preview: What the Assignment Asks You to Do

The assignment (hw8.ipynb) walks you through most of this same pipeline in a fill-in-the-blank format.

| Section | Task | Key methods |
|---------|------|-------------|
| 1 | Inspect the raw dataset; answer questions about NAs and dtypes | `pd.read_csv`, `isnull().sum()`, `dtypes` |
| 2 | Standardize text and remap smoker abbreviations | `str.strip().str.title()`, `map()` |
| 3 | Fill missing values; convert age back to int | `fillna(median)`, `astype(int)` |
| 4 | Derive age_group with apply(); count values | `apply()`, `value_counts()` |

The merge step is a demo-only preview for today. You will use `pd.merge()` in Module 9 and in your project.

After each step, verify your work before moving on:
- After filling NAs: `df.isnull().sum()`
- After standardizing text: `df['col'].unique()`
- After dedup: `len(df)` (should be 16)
- After apply: `df['age_group'].value_counts()`